In [ ]:
import numpy as np
from pandocfilters import attributes
from scipy.cluster.hierarchy import linkage

%load_ext autoreload
%autoreload 2

In [ ]:
import infoclus

In [ ]:
import os
from config import PROJECT_ROOT
from src.caching import from_cache

emb_name = 'tsne'
linkage = 'single'
modify_hierarchical = True
data_name = 'Mouse1_Batch1_WT_5k'
file_path = os.path.join(PROJECT_ROOT, 'data', data_name, 'cache', emb_name +'_'+ linkage + '_' + 'modify_' + str(modify_hierarchical))

if os.path.exists(file_path):
    print('loading ' + file_path)
    infoclus_obj = from_cache(file_path)
    print('done')
else:
    infoclus_obj = infoclus.InfoClus(dataset_name=data_name, emb_name=emb_name, linkage=linkage, modify_hierarchical=modify_hierarchical)

In [ ]:
infoclus_obj.optimise()
a = 10

In [ ]:
from sklearn.discriminant_analysis import StandardScaler
import infoclus_utils
import numpy as np

data = infoclus_obj.data_obj.data_raw.values
scaler = StandardScaler()
data = scaler.fit_transform(data)
prior = [np.mean(data, axis=0), np.var(data, axis=0)]
# prior = infoclus_obj.model_obj.prior
clustering = infoclus_obj.result_obj.clustering
attributes = infoclus_obj.result_obj.attributes_opt
count_clusters = infoclus_obj.result_obj.count_clusters
ic = 0

for i in range(count_clusters):
    cluster = data[np.where(clustering == i)]
    mean_cluster = np.mean(cluster, axis=0)
    var_cluster = np.var(cluster, axis=0)
    kl_cluster = infoclus_utils.kl_gaussian(mean_cluster, var_cluster, prior[0], prior[1])
    for attr in attributes[i]:
        ic = ic + len(cluster) * kl_cluster[attr]

c = infoclus_obj.alpha + (2*sum(len(att) for att in attributes))**infoclus_obj.beta
si = ic/c

si